# Phishing URL Risk Checker
**Author:** Gokul S  
A simple educational project that scores URLs using basic warning signs. It does not open URLs or guarantee that a website is safe.


## 1. Import required Python modules
This project uses only Python's standard library, so no third-party packages are required.


In [ ]:
from urllib.parse import urlparse
import ipaddress

SUSPICIOUS_KEYWORDS = ('login', 'verify', 'update', 'secure', 'account', 'password', 'banking', 'wallet', 'confirm', 'signin')
SHORTENER_DOMAINS = {'bit.ly', 'tinyurl.com', 't.co', 'is.gd', 'cutt.ly', 'ow.ly'}


## 2. Create the URL checking function


In [ ]:
def check_url(url):
    url = url.strip()
    reasons, score = [], 0
    if not url:
        return {'url': url, 'score': 0, 'risk': 'Invalid', 'reasons': ['No URL entered']}
    candidate = url if '://' in url else 'http://' + url
    parsed = urlparse(candidate)
    hostname = (parsed.hostname or '').lower()
    if not hostname:
        return {'url': url, 'score': 0, 'risk': 'Invalid', 'reasons': ['Could not identify a hostname']}
    if parsed.scheme.lower() != 'https':
        score += 1; reasons.append('URL does not use HTTPS')
    if '@' in parsed.netloc:
        score += 2; reasons.append("Contains '@' in the network location")
    if len(url) > 75:
        score += 1; reasons.append('Unusually long URL')
    if hostname.count('.') >= 3:
        score += 1; reasons.append('Many subdomains')
    try:
        ipaddress.ip_address(hostname)
        score += 2; reasons.append('Uses an IP address instead of a domain name')
    except ValueError:
        pass
    if hostname in SHORTENER_DOMAINS:
        score += 2; reasons.append('Uses a known URL-shortening domain')
    found = [word for word in SUSPICIOUS_KEYWORDS if word in url.lower()]
    if found:
        score += 1; reasons.append('Contains account-related keywords: ' + ', '.join(found))
    if '-' in hostname:
        score += 1; reasons.append('Domain contains a hyphen')
    risk = 'High' if score >= 4 else ('Medium' if score >= 2 else 'Low')
    if not reasons:
        reasons.append('No common warning signs found by these simple rules')
    return {'url': url, 'score': score, 'risk': risk, 'reasons': reasons}


## 3. Test with sample URLs
These are examples for demonstrating the rules. A low score does not prove a URL is safe.


In [ ]:
sample_urls = [
    'https://www.wikipedia.org',
    'http://secure-account-login.example.com/verify',
    'http://192.0.2.10/login',
    'https://bit.ly/example'
]
for item in sample_urls:
    result = check_url(item)
    print(f"URL: {result['url']}\nRisk: {result['risk']} (score {result['score']})")
    for reason in result['reasons']:
        print(' -', reason)
    print('-' * 50)


## 4. Check your own URL
Enter only URLs you are comfortable analyzing. The program checks the text locally and does not connect to the website.


In [ ]:
url_to_check = input('Enter a URL: ')
result = check_url(url_to_check)
print('Risk:', result['risk'])
print('Score:', result['score'])
print('Reasons:')
for reason in result['reasons']:
    print('-', reason)
